> [!WARNING]
> **METADATA BLOCKED SUBMISSION WORKFLOW**:
> Official Kaggle `test.csv` rows contain only `sample_id`, `image_path`, and `target_name`. They lack all training metadata (`State`, `Species`, `NDVI`, `Height`, `Sampling_Date`). This model cannot generate standard test predictions without metadata. See `notebooks/evaluation/unified_metadata_evaluation.ipynb` for counterfactual evaluation.


> [!WARNING]
> **OFFLINE SUBMISSION NOTEBOOK — REQUIRES SPECIAL DEPENDENCIES**
> Internet is disabled. Upload the following as Kaggle datasets:
> 1. `b6-vmamba-checkpoints` — fold checkpoints (`fold0_best.pth` ... `fold4_best.pth`)
> 2. `mamba-deps` — containing `causal_conv1d-1.6.0.tar.gz`, `mamba_ssm-2.3.0.tar.gz`, and `vmamba.py`
>
> The mamba-ssm packages will be compiled from source (~5-10 min with GPU enabled).


# B6: VMamba-Base + Mamba SSM Fusion + Metadata MLP

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

Inference notebook for **VMamba-Base** (~89M params) with 2D Selective State Space Model
backbone and Mamba SSM fusion blocks. Includes metadata MLP fusion (23 features).

- **Backbone:** VMamba-Base (VSSM, depths=[2,2,15,2], dims=128, 1024-d output)
- **Fusion:** 2× MambaFusionBlock (real Mamba SSM CUDA kernels)
- **Metadata:** State one-hot + Species one-hot + NDVI + Height + month sin/cos (23 features)
- **5-fold ensemble** with compositional regression heads (Softplus)


## Inference and Submission Generation (VMamba)

In [ ]:
# --- Fail Fast Schema Check for Official Kaggle Test Data ---
import pandas as pd
import os

test_csv_candidates = [
    '/kaggle/input/csiro-biomass/test.csv',
    '/kaggle/input/competitions/csiro-biomass/test.csv',
    'csiro-biomass/test.csv',
]
test_csv_path = next((p for p in test_csv_candidates if os.path.isfile(p)), None)
if test_csv_path is not None:
    test_df = pd.read_csv(test_csv_path)
    required_meta = ['State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'Sampling_Date']
    missing = [c for c in required_meta if c not in test_df.columns]
    if missing:
        raise ValueError(
            f"Official test.csv lacks required metadata columns: {missing}. "
            "This model was trained with 23 tabular metadata features and is metadata blocked "
            "by the official competition test schema. Do not silently fill zeros or fabricate metadata. "
            "See notebooks/evaluation/unified_metadata_evaluation.ipynb for zero metadata counterfactual evaluation."
        )


In [ ]:
# --- Setup: install mamba-ssm, causal-conv1d, and vmamba from uploaded dataset ---
import subprocess, sys, shutil, os
import glob as _glob

def _find(slug, pattern=None):
    """Find Kaggle dataset dir. If pattern given, searches subdirs too."""
    for base in [f'/kaggle/input/{slug}', *_glob.glob(f'/kaggle/input/datasets/*/{slug}')]:
        if not os.path.isdir(base): continue
        if pattern:
            if _glob.glob(os.path.join(base, pattern)): return base
            for sub in _glob.glob(os.path.join(base, '*')):
                if os.path.isdir(sub) and _glob.glob(os.path.join(sub, pattern)):
                    return sub
        return base
    raise FileNotFoundError(f"Dataset '{slug}' not found in /kaggle/input/")


MAMBA_DEPS = _find('mamba-deps')

# 1. Install causal-conv1d (dependency of mamba-ssm) — builds CUDA kernels
print("Installing causal-conv1d from source (compiling CUDA kernels)...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    os.path.join(MAMBA_DEPS, 'causal_conv1d-1.6.0.tar.gz'),
    '--no-build-isolation', '-q'])

# 2. Install mamba-ssm — builds CUDA kernels
print("Installing mamba-ssm from source (compiling CUDA kernels)...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    os.path.join(MAMBA_DEPS, 'mamba_ssm-2.3.0.tar.gz'),
    '--no-build-isolation', '-q'])

# 3. Copy vmamba.py to working directory so it can be imported
shutil.copy(os.path.join(MAMBA_DEPS, 'vmamba.py'), '/kaggle/working/vmamba.py')
sys.path.insert(0, '/kaggle/working')

print("All mamba dependencies installed successfully")


In [ ]:
import os, gc, sys, warnings
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings('ignore')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
class CFG:
    BASE_PATH = '/kaggle/input/competitions/csiro-biomass'
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    # Upload fold checkpoints as a Kaggle dataset:
    MODEL_DIR = _find('b6-vmamba-checkpoints', '*.pth')
    # Upload VMamba pretrained weights (for model architecture init):
    VMAMBA_PRETRAINED = None  # Not needed at inference (fold checkpoint has all weights)
    MODEL_NAME = 'vmamba_base'
    N_FOLDS = 5
    FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
    IMG_SIZE = 512
    BATCH_SIZE = 4
    NUM_WORKERS = 0
    DROPOUT = 0.2
    USE_METADATA = True
    META_INPUT_DIM = 23
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {CFG.DEVICE}, Model: {CFG.MODEL_NAME}")
print(f"Checkpoints: {CFG.MODEL_DIR}")


In [ ]:
# --- VMamba + Mamba Model Architecture ---
# These must match training exactly.

from mamba_ssm import Mamba
from vmamba import VSSM

class MambaFusionBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dropout=0.1, **kwargs):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.drop = nn.Dropout(dropout)

    @torch.compiler.disable
    def forward(self, x):
        shortcut = x
        with torch.amp.autocast('cuda', enabled=False):
            x = x.float()
            x = self.norm(x)
            x = self.mamba(x)
            x = self.drop(x)
        return shortcut + x

def _make_head(nf, dropout):
    return nn.Sequential(
        nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(nf//2, 1), nn.Softplus()
    )

class VMambaBackbone(nn.Module):
    def __init__(self, pretrained_path='', variant='vmamba_base'):
        super().__init__()
        CONFIGS = {
            'vmamba_base': {
                'depths': [2,2,15,2], 'dims': 128, 'num_features': 1024,
                'vssm_kwargs': dict(
                    ssm_d_state=1, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer="silu",
                    ssm_conv=3, ssm_conv_bias=False, ssm_init="v0", forward_type="v05_noz",
                    mlp_ratio=4.0, mlp_act_layer="gelu", patch_norm=True, norm_layer="ln2d",
                    downsample_version="v3", patchembed_version="v2",
                ),
            }
        }
        cfg = CONFIGS[variant]
        self.model = VSSM(depths=cfg['depths'], dims=cfg['dims'], **cfg['vssm_kwargs'])
        self.num_features = cfg['num_features']
        self._is_v0 = False  # v2 checkpoint
        if pretrained_path and os.path.exists(pretrained_path):
            ckpt = torch.load(pretrained_path, map_location='cpu', weights_only=False)
            if 'model' in ckpt: ckpt = ckpt['model']
            ckpt = {k: v for k, v in ckpt.items() if not k.startswith('classifier.')}
            self.model.load_state_dict(ckpt, strict=False)
            print(f"Loaded VMamba weights from {pretrained_path}")

    @torch.compiler.disable
    def forward(self, x):
        x = self.model.patch_embed(x)
        if self.model.pos_embed is not None:
            x = x + self.model.pos_embed
        for layer in self.model.layers:
            x = layer(x)
        if hasattr(self.model.classifier, 'norm'):
            x = self.model.classifier.norm(x)
        elif hasattr(self.model, 'norm'):
            x = self.model.norm(x)
        if x.dim() == 4:
            B, C, H, W = x.shape  # v2: channel-first
            x = x.permute(0, 2, 3, 1).reshape(B, H*W, C)
        return x

class BiomassModelVMamba(nn.Module):
    def __init__(self, pretrained_path='', dropout=0.2, use_mamba_ssm=True,
                 use_metadata=False, meta_input_dim=23, variant='vmamba_base'):
        super().__init__()
        self.backbone = VMambaBackbone(pretrained_path, variant=variant)
        nf = self.backbone.num_features
        FusionBlock = MambaFusionBlock if use_mamba_ssm else None
        self.fusion = nn.Sequential(FusionBlock(nf, dropout=dropout), FusionBlock(nf, dropout=dropout))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.use_metadata = use_metadata
        if use_metadata:
            meta_hidden = 64
            self.meta_mlp = nn.Sequential(
                nn.Linear(meta_input_dim, meta_hidden), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(meta_hidden, meta_hidden),
            )
            self.meta_proj = nn.Sequential(nn.Linear(nf + meta_hidden, nf), nn.GELU())
        self.head_green = _make_head(nf, dropout)
        self.head_dead = _make_head(nf, dropout)
        self.head_clover = _make_head(nf, dropout)

    def forward(self, left, right, metadata=None):
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x = torch.cat([x_l, x_r], dim=1)
        x = self.fusion(x)
        x = self.pool(x.transpose(1, 2)).flatten(1)
        if self.use_metadata and metadata is not None:
            meta_feat = self.meta_mlp(metadata)
            x = self.meta_proj(torch.cat([x, meta_feat], dim=1))
        green = self.head_green(x); dead = self.head_dead(x); clover = self.head_clover(x)
        gdm = green + clover; total = gdm + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print("VMamba model architecture defined")


In [ ]:
# --- Metadata Encoding ---
def encode_metadata(df):
    """Encode metadata features for test images. Returns (meta_array, meta_dim)."""
    meta = df.copy()
    
    # One-hot: State (4 categories in training)
    STATES = ['NSW', 'QLD', 'TAS', 'VIC']
    for s in STATES:
        meta[f'meta_state_{s}'] = (meta['State'] == s).astype(np.float32)
    
    # One-hot: Species (15 categories in training)
    SPECIES = [
        'Brachiaria decumbens', 'Chloris gayana', 'Digitaria eriantha',
        'Festuca arundinacea', 'Lolium multiflorum', 'Lolium perenne',
        'Megathyrsus maximus', 'Mixed', 'Paspalum dilatatum',
        'Pennisetum clandestinum', 'Setaria sphacelata',
        'Trifolium repens/Lolium perenne', 'Trifolium subterraneum',
        'Trifolium subterraneum/Lolium perenne',
        'Trifolium subterraneum/Phalaris aquatica'
    ]
    for sp in SPECIES:
        meta[f'meta_species_{sp}'] = (meta['Species'] == sp).astype(np.float32)
    
    # Continuous
    meta['meta_ndvi'] = meta['Pre_GSHH_NDVI'].astype(np.float32)
    meta['meta_height'] = (meta['Height_Ave_cm'] / 100.0).astype(np.float32)
    
    # Cyclical month
    month = pd.to_datetime(meta['Sampling_Date']).dt.month
    meta['meta_month_sin'] = np.sin(2 * np.pi * month / 12).astype(np.float32)
    meta['meta_month_cos'] = np.cos(2 * np.pi * month / 12).astype(np.float32)
    
    meta_cols = ([f'meta_state_{s}' for s in STATES] +
                 [f'meta_species_{sp}' for sp in SPECIES] +
                 ['meta_ndvi', 'meta_height', 'meta_month_sin', 'meta_month_cos'])
    
    meta[meta_cols] = meta[meta_cols].fillna(0.0)
    print(f"Metadata: {len(meta_cols)} features")
    return meta[meta_cols].values.astype(np.float32), len(meta_cols)

print("Metadata encoder defined")


In [ ]:
# --- Test Data & Inference ---
def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

class TestBiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, meta_array=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
        self.meta = meta_array
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img = cv2.imread(path)
        if img is None: img = np.zeros((1000,2000,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape; mid = w//2
        left, right = img[:, :mid], img[:, mid:]
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        if self.meta is not None:
            return left, right, torch.tensor(self.meta[idx], dtype=torch.float32)
        return left, right

# Load test data
test_long = pd.read_csv(CFG.TEST_CSV)
test_long['image_id'] = test_long['sample_id'].str.split('__').str[0]
test_df = test_long.drop_duplicates('image_id').reset_index(drop=True)
print(f"Test images: {len(test_df)}")

# Encode metadata
meta_array, meta_dim = encode_metadata(test_df)
print(f"Metadata shape: {meta_array.shape}")

test_dataset = TestBiomassDataset(test_df, CFG.TEST_IMAGE_DIR, get_val_transforms(), meta_array)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=True)

@torch.no_grad()
def predict_test(model, loader, device, use_meta=True):
    model.eval(); all_preds = []
    for batch in tqdm(loader, desc='Inference'):
        if use_meta:
            left, right, meta = batch
            meta = meta.to(device)
        else:
            left, right = batch
            meta = None
        with autocast('cuda'):
            preds = model(left.to(device), right.to(device), metadata=meta)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds)

# Ensemble across folds
all_fold_preds = []
for fold in CFG.FOLDS_TO_TRAIN:
    ckpt_path = os.path.join(CFG.MODEL_DIR, f'fold{fold}_best.pth')
    if not os.path.exists(ckpt_path):
        print(f'Checkpoint not found: {ckpt_path}, skipping fold {fold}.')
        continue
    # Create model without pretrained weights (we load fold checkpoint)
    model = BiomassModelVMamba(
        pretrained_path='', dropout=CFG.DROPOUT, use_mamba_ssm=True,
        use_metadata=CFG.USE_METADATA, meta_input_dim=meta_dim
    ).to(CFG.DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE, weights_only=True))
    print(f'Loaded fold {fold}')
    all_fold_preds.append(predict_test(model, test_loader, CFG.DEVICE, use_meta=CFG.USE_METADATA))
    del model; gc.collect(); torch.cuda.empty_cache()

if not all_fold_preds:
    raise RuntimeError('No fold checkpoints found.')
avg_preds = np.mean(all_fold_preds, axis=0)
print(f'Ensemble of {len(all_fold_preds)} folds, shape: {avg_preds.shape}')


In [ ]:
# --- Submission ---
image_ids = test_df['image_id'].values
pred_map = {}
for i, img_id in enumerate(image_ids):
    for j, tn in enumerate(CFG.TARGET_COLS):
        pred_map[(img_id, tn)] = float(avg_preds[i, j])

test_long['target'] = test_long.apply(
    lambda row: pred_map.get((row['image_id'], row['target_name']), 0.0), axis=1)
df_sub = test_long[['sample_id','target']].copy()

sample_sub = pd.read_csv(os.path.join(CFG.BASE_PATH, 'sample_submission.csv'))
df_sub = sample_sub[['sample_id']].merge(df_sub, on='sample_id', how='left')
df_sub['target'] = df_sub['target'].fillna(0.0)
df_sub.to_csv('submission.csv', index=False)
print(f'Saved submission.csv, shape: {df_sub.shape}')
print(df_sub.head(10).to_string(index=False))
